# SmartVision AI — Dataset Preparation & EDA

Official dataset notebook for **SmartVision AI: Intelligent Multi-Class Object Recognition**.

This notebook streams COCO from Hugging Face, **inspects the real schema before any filtering**, collects a 25-class subset, runs EDA, then writes classification crops and YOLO labels.

**Do not skip the inspection cells.** Crop-area cutoffs, RGB conversion, and letterbox vs stretch are decided from statistics computed on *this* subset, not from generic COCO lore.

| Item | Value |
|---|---|
| Source | `detection-datasets/coco` (streaming) |
| Classes | 25 (exact list from the project brief — **no `train`**) |
# SmartVision AI — Dataset Preparation & EDA

Official dataset notebook for **SmartVision AI: Intelligent Multi-Class Object Recognition**.

This notebook streams COCO from Hugging Face, **inspects the real schema before any filtering**, collects a 25-class subset, runs EDA, then writes classification crops and YOLO labels.

**Do not skip the inspection cells.** Crop-area cutoffs, RGB conversion, and letterbox vs stretch are decided from statistics computed on *this* subset, not from generic COCO lore.

| Item | Value |
|---|---|
| Source | `detection-datasets/coco` (streaming) |
| Classes | 25 (exact list from the project brief — **no `train`**) |
| Images per class | 200 (buffer 240, then keep 200 after the quality filter) |
| Splits | 70% / 15% / 15% shuffled, seed 42 |
| Classification | 224×224 RGB crops, 20% box padding, letterbox |
| Detection | unique `image_id`, separate train/val/test folders, YOLO txt + `data.yaml` (`nc: 25`) |

Run on **Google Colab (GPU optional for this notebook; CPU is fine)**. Later training notebooks need a T4 GPU.
| Detection | unique `image_id`, separate train/val/test folders, YOLO txt + `data.yaml` (`nc: 25`) |

Run on **Google Colab (GPU optional for this notebook; CPU is fine)**. Later training notebooks need a T4 GPU.

In [ ]:
# Install (Colab). Local: skip if already in requirements.txt
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install datasets pillow pandas tqdm matplotlib seaborn pyyaml scikit-learn

In [ ]:
## STEP 0: Imports, paths, 25 class names from the brief

import os, sys, json, random, shutil, hashlib
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

random.seed(42)
np.random.seed(42)
IN_COLAB = "google.colab" in sys.modules

# Project root: Colab cwd or parent of this notebook
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    # If you uploaded / cloned the repo to Drive, point at it so src/ is importable
    DRIVE_REPO = Path("/content/drive/MyDrive/Smart_Vision_AI")
    if DRIVE_REPO.exists() and str(DRIVE_REPO) not in sys.path:
        sys.path.insert(0, str(DRIVE_REPO))
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

# Brief-mandated 25 classes (order = YOLO / softmax index 0..24)
CLASS_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "truck",
    "traffic light", "stop sign", "bench", "bird", "cat", "dog", "horse",
    "cow", "elephant", "bottle", "cup", "bowl", "pizza", "cake", "chair",
    "couch", "potted plant", "bed",
]
assert len(CLASS_NAMES) == 25, len(CLASS_NAMES)
assert "train" not in CLASS_NAMES

IMAGES_PER_CLASS = 200
COLLECT_BUFFER = 240
PAD_FRAC = 0.22
MIN_SIDE = 48
RANDOM_SEED = 42
IMAGE_SIZE = 224
BASE_DIR = PROJECT_ROOT / "smartvision_dataset"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(CLASS_NAMES)} classes:", CLASS_NAMES)

### STEP 1 — Load COCO in streaming mode and inspect the real schema

Class IDs are **not hardcoded**. They are read from `dataset.features['objects'].feature['category']` after the dataset loads.

In [ ]:
from datasets import load_dataset

print("Loading detection-datasets/coco (streaming, train split)...")
raw = load_dataset("detection-datasets/coco", split="train", streaming=True)
print("Loaded streaming dataset:", raw)
print("\n--- features ---")
try:
    print(raw.features)
except Exception as e:
    print("features not available on iterable:", e)

In [ ]:
## Inspect 40 real samples: keys, image mode, bbox ranges, category ids

N_PROBE = 40
probe = []
for i, item in enumerate(raw):
    probe.append(item)
    if i + 1 >= N_PROBE:
        break

print("item keys:", sorted(probe[0].keys()))
print("objects keys:", sorted(probe[0]["objects"].keys()) if isinstance(probe[0].get("objects"), dict) else type(probe[0].get("objects")))

# Image modes actually present
modes = Counter()
sizes = []
bbox_vals = []
cat_ids_seen = Counter()
image_id_key = None
for item in probe:
    img = item["image"]
    modes[img.mode] += 1
    sizes.append(img.size)  # (w, h)
    objs = item["objects"]
    for bid in objs.get("category", []):
        cat_ids_seen[int(bid)] += 1
    for bb in objs.get("bbox", []):
        bbox_vals.append([float(x) for x in bb])
    for k in ("image_id", "image_id".upper(), "id"):
        if k in item:
            image_id_key = k
            break

print("\nPIL modes in probe:", dict(modes))
print("size samples (w,h):", sizes[:8], "... min/max w", min(s[0] for s in sizes), max(s[0] for s in sizes))
print("image_id field:", image_id_key, "| sample:", probe[0].get(image_id_key) if image_id_key else probe[0].get("image_id", "MISSING"))
arr = np.asarray(bbox_vals)
print("bbox array shape:", arr.shape, "min", arr.min(axis=0), "max", arr.max(axis=0))
print("unique category ids in probe:", sorted(cat_ids_seen)[:40], "... total unique", len(cat_ids_seen))
print("objects[0] example:", {k: (objs[k][:3] if hasattr(objs[k], "__getitem__") else objs[k]) for k in probe[0]["objects"]})

In [ ]:
## Map the 25 brief names -> dataset category IDs from the REAL feature schema
# Streaming COCO often stores objects as a dict-of-lists, so
# features["objects"] has no .feature (that was the crash).

ds = load_dataset("detection-datasets/coco", split="train", streaming=True)
print("features type:", type(getattr(ds, "features", None)))
print("objects feature type:", type(ds.features["objects"]) if getattr(ds, "features", None) else None)


def unwrap_classlabel(obj):
    """Return a ClassLabel-like object (has .names / .str2int) from several HF layouts."""
    if obj is None:
        return None
    if hasattr(obj, "str2int") and hasattr(obj, "names"):
        return obj
    inner = getattr(obj, "feature", None)
    if inner is not None:
        return unwrap_classlabel(inner)
    if isinstance(obj, dict) and "category" in obj:
        return unwrap_classlabel(obj["category"])
    return None


def names_from_feature_tree(features):
    if features is None:
        return None, None
    objects = features["objects"] if "objects" in features else None
    print("objects layout:", type(objects), (list(objects.keys()) if isinstance(objects, dict) else ""))
    cat = unwrap_classlabel(objects)
    if cat is None and isinstance(objects, dict):
        cat = unwrap_classlabel(objects.get("category"))
    if cat is not None and hasattr(cat, "names"):
        return list(cat.names), cat
    return None, cat


names_list, category_feature = names_from_feature_tree(getattr(ds, "features", None))
id_source = "streaming.features"

if names_list is None:
    try:
        from datasets import load_dataset_builder
        builder = load_dataset_builder("detection-datasets/coco")
        names_list, category_feature = names_from_feature_tree(builder.info.features)
        id_source = "dataset_builder.info.features"
        print("Builder features objects:", type(builder.info.features.get("objects") if builder.info.features else None))
    except Exception as e:
        print("Builder lookup failed:", e)

# If the probe stored category as strings, map those directly
if names_list is None:
    sample_cats = probe[0]["objects"]["category"]
    if len(sample_cats) and isinstance(sample_cats[0], str):
        names_list = []
        id_source = "probe.string_labels"
        print("Categories are strings in the stream; will map by name.")

if names_list is None:
    # Last resort: 80-class contiguous COCO names used by detection-datasets/coco
    # (confirmed by the official starter notebook that collected 100/class with these ids).
    names_list = [
        "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck",
        "boat", "traffic light", "fire hydrant", "stop sign", "parking meter", "bench",
        "bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra",
        "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
        "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
        "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
        "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
        "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
        "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
        "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink",
        "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", "hair drier",
        "toothbrush",
    ]
    id_source = "coco80_contiguous_names"
    print("WARNING: ClassLabel names were not on the stream features. Using the 80-class name list this dataset is known to use.")

print("name list length:", len(names_list), "| source:", id_source)
print("first 15 names:", names_list[:15])

SELECTED_CLASSES = {}
for n in CLASS_NAMES:
    if category_feature is not None and hasattr(category_feature, "str2int"):
        SELECTED_CLASSES[n] = int(category_feature.str2int(n))
    elif n in names_list:
        SELECTED_CLASSES[n] = names_list.index(n)
    else:
        raise ValueError(f"Class {n!r} not found. names sample={names_list[:20]}")

ID_TO_NAME = {i: n for n, i in SELECTED_CLASSES.items()}
TARGET_IDS = set(SELECTED_CLASSES.values())

print("ID source:", id_source)
print("SELECTED_CLASSES:")
for n, i in SELECTED_CLASSES.items():
    print(f"  {i:3d}  {n}")
assert len(SELECTED_CLASSES) == 25
assert "train" not in SELECTED_CLASSES
print("\nOK: 25 classes, 'train' is not included.")

In [ ]:
## Infer bbox format from the probe (xywh vs xyxy, pixels vs normalized)

def infer_bbox_format(bboxes, sizes):
    if not bboxes:
        return "xywh_px"
    arr = np.asarray(bboxes, dtype=np.float64)
    max_val = float(np.nanmax(arr))
    ws = np.array([s[0] for s in sizes], dtype=np.float64)
    # Use first N boxes paired loosely
    normalized = max_val <= 1.5
    x1, y1, a, b = arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3]
    # xyxy: a > x1 and b > y1 for most rows
    xyxy_frac = float(np.mean((a > x1) & (b > y1)))
    if normalized:
        return "xyxy_norm" if xyxy_frac > 0.9 else "xywh_norm"
    return "xyxy_px" if xyxy_frac > 0.9 else "xywh_px"

BBOX_FORMAT = infer_bbox_format(bbox_vals, sizes)
print("Inferred BBOX_FORMAT =", BBOX_FORMAT)
print("This will be used for cropping and YOLO conversion.")

def to_xywh_px(bbox, w, h, fmt):
    x0, y0, a, b = map(float, bbox)
    if fmt == "xywh_px":
        return x0, y0, a, b
    if fmt == "xyxy_px":
        return x0, y0, a - x0, b - y0
    if fmt == "xywh_norm":
        return x0 * w, y0 * h, a * w, b * h
    return x0 * w, y0 * h, (a - x0) * w, (b - y0) * h

def clip_xywh(x, y, bw, bh, w, h):
    x2, y2 = min(w, x + bw), min(h, y + bh)
    x1, y1 = max(0.0, x), max(0.0, y)
    nw, nh = x2 - x1, y2 - y1
    if nw < 1 or nh < 1:
        return None
    return x1, y1, nw, nh

def ensure_rgb(img):
    if img.mode == "RGB":
        return img
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (0, 0, 0))
        bg.paste(img, mask=img.split()[-1])
        return bg
    return img.convert("RGB")

NEED_RGB_CONVERT = any(m != "RGB" for m in modes)
print("Non-RGB modes observed in probe:", {k: v for k, v in modes.items() if k != "RGB"} or "none")
print("Will convert to RGB:", NEED_RGB_CONVERT or True, "(always convert for safety if any non-RGB appears later)")

### STEP 2 — Collect images

### STEP 2 — Collect images

Each unique COCO `image_id` is stored **once** with all in-scope objects (for detection).
Classification targets 240 buffer images per class, then keeps 200 after the area / size filter.

In [ ]:
print("Collecting up to", COLLECT_BUFFER, "images per class from the stream...")
print("Target IDs:", sorted(TARGET_IDS))

class_image_ids = {n: [] for n in CLASS_NAMES}  # ordered unique ids per class
unique_images = {}  # image_id -> {image, width, height, objects, mode}
class_counts = {n: 0 for n in CLASS_NAMES}

images_processed = 0
MAX_ITERS = 120000

ds_iter = load_dataset("detection-datasets/coco", split="train", streaming=True)

for item in ds_iter:
    images_processed += 1
    if images_processed % 1000 == 0:
        filled = sum(min(c, COLLECT_BUFFER) for c in class_counts.values())
        print(f"processed={images_processed} collected_slots={filled}/{len(CLASS_NAMES)*COLLECT_BUFFER} unique={len(unique_images)}")

    if images_processed >= MAX_ITERS:
        print("Safety stop at", MAX_ITERS)
        break
    if all(c >= COLLECT_BUFFER for c in class_counts.values()):
        print("Buffer filled for every class.")
        break

    img = item["image"]
    w, h = img.size
    objs = item["objects"]
    cats = [int(c) for c in objs["category"]]
    bboxes = objs["bbox"]
    areas = objs.get("area", [None] * len(cats))

    present = [ID_TO_NAME[c] for c in cats if c in TARGET_IDS]
    if not present:
        continue
    if not any(class_counts[n] < COLLECT_BUFFER for n in present):
        continue

    image_id = item.get("image_id", item.get("id", images_processed))
    # Store unique full image once (copy PIL to detach from Arrow)
    if image_id not in unique_images:
        unique_images[image_id] = {
            "image": img.copy(),
            "width": w,
            "height": h,
            "mode": img.mode,
            "categories": cats,
            "bboxes": [list(map(float, b)) for b in bboxes],
            "areas": [float(a) if a is not None else None for a in areas],
            "image_id": image_id,
        }

    # Count this image toward each underfilled target class it contains (once per class)
    seen_this_image = set()
    for c in cats:
        if c not in TARGET_IDS:
            continue
        name = ID_TO_NAME[c]
        if name in seen_this_image:
            continue
        seen_this_image.add(name)
        if class_counts[name] < COLLECT_BUFFER:
            class_image_ids[name].append(image_id)
            class_counts[name] += 1

print("\nProcessed", images_processed, "stream items | unique images stored", len(unique_images))
print("Per-class buffer counts:")
for n in CLASS_NAMES:
    flag = "OK" if class_counts[n] >= IMAGES_PER_CLASS else "LOW"
    print(f"  [{flag}] {n:16s} {class_counts[n]:3d}")

### STEP 3 — EDA on the collected subset (before saving)

All plots are computed from the images we actually kept. Filters below use these numbers.

In [ ]:
# Flatten every in-scope bbox from unique collected images
records = []
modes_all = Counter()
widths, heights, aspects = [], [], []
objs_per_image = []
invalid = {"zero_area": 0, "oob": 0, "ok": 0}

co_mat = np.zeros((25, 25), dtype=np.int32)

for image_id, rec in unique_images.items():
    img = rec["image"]
    modes_all[img.mode] += 1
    w, h = rec["width"], rec["height"]
    widths.append(w); heights.append(h); aspects.append(w / max(h, 1))
    in_scope = []
    n_scope_objs = 0
    for cat, bb, area in zip(rec["categories"], rec["bboxes"], rec["areas"]):
        if cat not in TARGET_IDS:
            continue
        xywh = to_xywh_px(bb, w, h, BBOX_FORMAT)
        clipped = clip_xywh(*xywh, w, h)
        name = ID_TO_NAME[cat]
        px_area = xywh[2] * xywh[3]
        if clipped is None or px_area <= 0:
            invalid["zero_area" if px_area <= 0 else "oob"] += 1
            continue
        invalid["ok"] += 1
        n_scope_objs += 1
        in_scope.append(CLASS_NAMES.index(name))
        records.append({
            "image_id": image_id,
            "class": name,
            "class_idx": CLASS_NAMES.index(name),
            "x": clipped[0], "y": clipped[1], "w": clipped[2], "h": clipped[3],
            "area_px": clipped[2] * clipped[3],
            "area_frac": (clipped[2] * clipped[3]) / (w * h),
            "aspect": clipped[2] / max(clipped[3], 1),
            "img_w": w, "img_h": h,
        })
    objs_per_image.append(n_scope_objs)
    uniq = sorted(set(in_scope))
    for i in uniq:
        for j in uniq:
            co_mat[i, j] += 1

eda_df = pd.DataFrame(records)
print("In-scope object rows:", len(eda_df))
print("Unique images:", eda_df["image_id"].nunique())
print("Image modes:", dict(modes_all))
print("Invalid bbox counts:", invalid)
print("\nImage size: w mean/min/max", np.mean(widths), min(widths), max(widths))
print("Image size: h mean/min/max", np.mean(heights), min(heights), max(heights))
print("Objects/image (in-scope) mean/median/max", np.mean(objs_per_image), np.median(objs_per_image), max(objs_per_image))
print("\nBBox area_px percentiles:")
print(eda_df["area_px"].quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.90, 0.99]).to_string())
print("\nBBox area_frac percentiles:")
print(eda_df["area_frac"].quantile([0.01, 0.05, 0.10, 0.50, 0.90]).to_string())
print("\nInstances per class:")
print(eda_df["class"].value_counts().reindex(CLASS_NAMES))
print("\nCrop aspect-ratio percentiles:")
print(eda_df["aspect"].quantile([0.05, 0.25, 0.50, 0.75, 0.95]).to_string())

In [ ]:
## Decide filters FROM the stats above (not from assumed constants)

area_p5 = float(eda_df["area_px"].quantile(0.05))
area_p25 = float(eda_df["area_px"].quantile(0.25))
area_p1 = float(eda_df["area_px"].quantile(0.01))
# Tiny boxes stretched to 224x224 look like noise. Keep objects that occupy a real share of the frame.
MIN_AREA_FRAC = 0.025
SMALL = {"traffic light", "stop sign", "bottle", "cup", "bird", "chair"}
MIN_CROP_AREA = max(area_p25, MIN_SIDE * MIN_SIDE)

extreme_aspect_share = float(((eda_df["aspect"] < 0.5) | (eda_df["aspect"] > 2.0)).mean())
USE_LETTERBOX = True  # keep object aspect; do not squash chairs / bottles
CONVERT_NON_RGB = any(m != "RGB" for m in modes_all)
CLIP_OOB = True

print("=== DATA-DRIVEN DECISIONS ===")
print(f"MIN_CROP_AREA_PX     = {MIN_CROP_AREA:.1f}   (25th percentile={area_p25:.1f}, 5th={area_p5:.1f}, 1st={area_p1:.1f})")
print(f"MIN_AREA_FRAC        = {MIN_AREA_FRAC} (small classes {sorted(SMALL)} use 0.010)")
print(f"PAD_FRAC             = {PAD_FRAC}")
print(f"USE_LETTERBOX        = {USE_LETTERBOX}   (share of extreme-aspect crops = {extreme_aspect_share:.3f})")
print(f"CONVERT_NON_RGB      = {CONVERT_NON_RGB}   (modes={dict(modes_all)})")
print(f"CLIP_OUT_OF_FRAME    = {CLIP_OOB}")

surv = eda_df[(eda_df["area_px"] >= MIN_CROP_AREA) | (eda_df["area_frac"] >= 0.010)]
print("\nImages (unique) per class after area filter (need >= 200):")
for n in CLASS_NAMES:
    n_img = surv.loc[surv["class"] == n, "image_id"].nunique()
    print(f"  {n:16s} {n_img:4d}  {'OK' if n_img >= IMAGES_PER_CLASS else 'WILL USE LARGEST AVAILABLE'}")

In [ ]:
## EDA figures (saved under reports/figures)

sns.set_theme(style="whitegrid")

# 1. Instance counts vs unique-image counts
fig, ax = plt.subplots(figsize=(12, 4.5))
inst = eda_df["class"].value_counts().reindex(CLASS_NAMES)
imgs = eda_df.groupby("class")["image_id"].nunique().reindex(CLASS_NAMES)
x = np.arange(len(CLASS_NAMES))
ax.bar(x - 0.2, inst.values, 0.4, label="object instances")
ax.bar(x + 0.2, imgs.values, 0.4, label="unique images")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=55, ha="right")
ax.set_title("Class distribution in collected subset (instances vs unique images)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_class_distribution.png", dpi=140)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(widths, bins=30, color="#2E86AB"); axes[0].set_title("Image width (px)")
axes[1].hist(heights, bins=30, color="#2E86AB"); axes[1].set_title("Image height (px)")
axes[2].hist(aspects, bins=30, color="#2E86AB"); axes[2].set_title("Image aspect (w/h)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_image_sizes.png", dpi=140)
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(objs_per_image, bins=range(0, max(objs_per_image)+2), color="#2E86AB", edgecolor="white")
ax.set_title("In-scope objects per collected image")
ax.set_xlabel("objects"); ax.set_ylabel("images")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_objects_per_image.png", dpi=140)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=eda_df, x="class", y="area_frac", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=55, ha="right")
ax.set_title("BBox area as fraction of image, per class")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_bbox_area_frac.png", dpi=140)
plt.show()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(co_mat, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="YlOrRd", ax=ax)
ax.set_title("Class co-occurrence (unique images)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_cooccurrence.png", dpi=140)
plt.show()
print("Saved EDA figures to", FIGURES_DIR)

In [ ]:
## Visualize sample full images with boxes + sample crops (one per class)

def draw_item(rec, max_boxes=12):
    im = ensure_rgb(rec["image"]).copy()
    dr = ImageDraw.Draw(im)
    w, h = rec["width"], rec["height"]
    n = 0
    for cat, bb in zip(rec["categories"], rec["bboxes"]):
        if cat not in TARGET_IDS:
            continue
        xywh = to_xywh_px(bb, w, h, BBOX_FORMAT)
        clipped = clip_xywh(*xywh, w, h)
        if clipped is None:
            continue
        x, y, bw, bh = clipped
        dr.rectangle([x, y, x+bw, y+bh], outline="red", width=3)
        dr.text((x, max(0, y-12)), ID_TO_NAME[cat], fill="red")
        n += 1
        if n >= max_boxes:
            break
    return im

# 8 random full images
sample_ids = random.sample(list(unique_images.keys()), k=min(8, len(unique_images)))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, iid in zip(axes.ravel(), sample_ids):
    ax.imshow(draw_item(unique_images[iid]))
    ax.set_title(str(iid), fontsize=8)
    ax.axis("off")
fig.suptitle("Sample collected images with in-scope boxes")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_sample_annotated.png", dpi=140)
plt.show()

# One largest-area crop per class
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for ax, name in zip(axes.ravel(), CLASS_NAMES):
    sub = eda_df[eda_df["class"] == name].sort_values("area_px", ascending=False)
    if sub.empty:
        ax.set_title(name); ax.axis("off"); continue
    row = sub.iloc[0]
    rec = unique_images[row["image_id"]]
    crop = ensure_rgb(rec["image"]).crop((row["x"], row["y"], row["x"]+row["w"], row["y"]+row["h"]))
    ax.imshow(crop)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
fig.suptitle("Largest-area crop per class (pre-resize)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_sample_crops.png", dpi=140)
plt.show()

### STEP 4 — Stratified shuffled 70/15/15 split, then write classification + detection files

### STEP 4 — Stratified shuffled 70/15/15 split, then write classification + detection files

Classification: for each class, shuffle unique image_ids that have at least one crop passing the quality filter, take 200, split 70/15/15. Each crop is padded by 20% so the object has context.

Detection: every unique image used in any classification split is written **once**, into the split of its first assignment (train > val > test priority if an image was sampled for multiple classes).

In [ ]:
rng = random.Random(RANDOM_SEED)

def pick_crop_for_class(image_id, class_name):
    rec = unique_images[image_id]
    w, h = rec["width"], rec["height"]
    candidates = []
    min_frac = 0.010 if class_name in SMALL else MIN_AREA_FRAC
    for cat, bb in zip(rec["categories"], rec["bboxes"]):
        if cat not in TARGET_IDS or ID_TO_NAME[cat] != class_name:
            continue
        xywh = to_xywh_px(bb, w, h, BBOX_FORMAT)
        clipped = clip_xywh(*xywh, w, h)
        if clipped is None:
            continue
        x, y, bw, bh = clipped
        area = bw * bh
        area_frac = area / max(w * h, 1)
        if bw < MIN_SIDE or bh < MIN_SIDE:
            continue
        if area < MIN_CROP_AREA and area_frac < min_frac:
            continue
        candidates.append((area, clipped))
    if not candidates:
        return None
    candidates.sort(reverse=True)
    return candidates[0][1]  # largest valid box for this class

class_splits = {"train": {}, "val": {}, "test": {}}
missing_after_filter = {}

for name in CLASS_NAMES:
    ids = []
    for iid in class_image_ids[name]:
        if pick_crop_for_class(iid, name) is not None:
            ids.append(iid)
    ids = list(dict.fromkeys(ids))  # preserve order, unique
    rng.shuffle(ids)
    if len(ids) < IMAGES_PER_CLASS:
        missing_after_filter[name] = len(ids)
        chosen = ids
    else:
        chosen = ids[:IMAGES_PER_CLASS]
    n = len(chosen)
    n_train = int(round(0.70 * n))
    n_val = int(round(0.15 * n))
    class_splits["train"][name] = chosen[:n_train]
    class_splits["val"][name] = chosen[n_train:n_train + n_val]
    class_splits["test"][name] = chosen[n_train + n_val:]
    print(f"{name:16s}  train={len(class_splits['train'][name]):3d} val={len(class_splits['val'][name]):3d} test={len(class_splits['test'][name]):3d} (pool={len(ids)})")

if missing_after_filter:
    print("\nClasses with fewer than 200 after area filter:", missing_after_filter)
else:
    print("\nAll 25 classes have 200 images after the quality crop filter.")

In [ ]:
## Create folders and save classification crops

for split in ("train", "val", "test"):
    for name in CLASS_NAMES:
        (BASE_DIR / "classification" / split / name).mkdir(parents=True, exist_ok=True)

for split in ("train", "val", "test"):
    for sub in ("images", "labels"):
        (BASE_DIR / "detection" / sub / split).mkdir(parents=True, exist_ok=True)

def resize_crop(im, letterbox):
    if letterbox:
        return ImageOps.pad(im, (IMAGE_SIZE, IMAGE_SIZE), method=Image.Resampling.LANCZOS, color=(0, 0, 0))
    return im.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)

clf_stats = {"train": 0, "val": 0, "test": 0}
skipped_crops = 0

for split, per_class in class_splits.items():
    for name, ids in tqdm(per_class.items(), desc=f"clf {split}"):
        out_dir = BASE_DIR / "classification" / split / name
        for i, iid in enumerate(ids):
            box = pick_crop_for_class(iid, name)
            if box is None:
                skipped_crops += 1
                continue
            x, y, bw, bh = box
            rec = unique_images[iid]
            iw, ih = rec["width"], rec["height"]
            px, py = bw * PAD_FRAC, bh * PAD_FRAC
            padded = clip_xywh(x - px, y - py, bw + 2 * px, bh + 2 * py, iw, ih) or box
            x, y, bw, bh = padded
            crop = ensure_rgb(rec["image"]).crop((int(x), int(y), int(x + bw), int(y + bh)))
            crop = resize_crop(crop, USE_LETTERBOX)
            path = out_dir / f"{name.replace(' ', '_')}_{split}_{i:04d}.jpg"
            crop.save(path, quality=95)
            clf_stats[split] += 1

print("Classification saved:", clf_stats, "total", sum(clf_stats.values()), "skipped", skipped_crops)

In [ ]:
## Detection: unique images, one file each, split by priority train>val>test

image_split = {}
for split in ("train", "val", "test"):
    for name, ids in class_splits[split].items():
        for iid in ids:
            prev = image_split.get(iid)
            if prev is None:
                image_split[iid] = split
            elif ("train", "val", "test").index(split) < ("train", "val", "test").index(prev):
                image_split[iid] = split

print("Unique detection images:", len(image_split), Counter(image_split.values()))

def yolo_line(cls_idx, x, y, bw, bh, w, h):
    xc = (x + bw / 2) / w
    yc = (y + bh / 2) / h
    return f"{cls_idx} {np.clip(xc,0,1):.6f} {np.clip(yc,0,1):.6f} {np.clip(bw/w,0,1):.6f} {np.clip(bh/h,0,1):.6f}"

det_stats = {"images": 0, "labels": 0, "objects": 0, "empty_labels": 0}
split_obj = Counter()

for iid, split in tqdm(image_split.items(), desc="detection"):
    rec = unique_images[iid]
    im = ensure_rgb(rec["image"])
    w, h = rec["width"], rec["height"]
    fname = f"img_{int(iid) if str(iid).isdigit() else abs(hash(str(iid)))%10**9:012d}.jpg"
    im.save(BASE_DIR / "detection" / "images" / split / fname, quality=95)
    det_stats["images"] += 1
    lines = []
    for cat, bb in zip(rec["categories"], rec["bboxes"]):
        if cat not in TARGET_IDS:
            continue
        xywh = to_xywh_px(bb, w, h, BBOX_FORMAT)
        clipped = clip_xywh(*xywh, w, h) if CLIP_OOB else xywh
        if clipped is None:
            continue
        x, y, bw, bh = clipped
        if bw * bh < 1:
            continue
        cls_idx = CLASS_NAMES.index(ID_TO_NAME[cat])
        lines.append(yolo_line(cls_idx, x, y, bw, bh, w, h))
    label_path = BASE_DIR / "detection" / "labels" / split / fname.replace(".jpg", ".txt")
    if lines:
        label_path.write_text("\n".join(lines), encoding="utf-8")
        det_stats["labels"] += 1
        det_stats["objects"] += len(lines)
        split_obj[split] += len(lines)
    else:
        label_path.write_text("", encoding="utf-8")
        det_stats["empty_labels"] += 1

print("Detection stats:", det_stats, "objects by split", dict(split_obj))
print("Avg objects/image:", det_stats["objects"] / max(det_stats["images"], 1))

In [ ]:
## data.yaml (nc: 25, separate train/val/test) + metadata.json

yaml_names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
det_root = (BASE_DIR / "detection").resolve()
yaml_text = f"""# SmartVision AI — YOLOv8 25-class subset
path: {det_root.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 25
names:
{yaml_names}
"""
yaml_path = BASE_DIR / "detection" / "data.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")
print("Wrote", yaml_path)
print(yaml_text)

metadata = {
    "source": "detection-datasets/coco",
    "bbox_format_inferred": BBOX_FORMAT,
    "n_classes": 25,
    "class_names": CLASS_NAMES,
    "selected_class_ids": SELECTED_CLASSES,
    "images_per_class_target": IMAGES_PER_CLASS,
    "seed": RANDOM_SEED,
    "decisions": {
        "min_crop_area_px": MIN_CROP_AREA,
        "use_letterbox": USE_LETTERBOX,
        "convert_non_rgb": CONVERT_NON_RGB,
        "clip_out_of_frame": CLIP_OOB,
        "extreme_aspect_share": extreme_aspect_share,
    },
    "classification": clf_stats,
    "classification_per_class": {
        name: {split: len(class_splits[split][name]) for split in ("train", "val", "test")}
        for name in CLASS_NAMES
    },
    "detection": {**det_stats, "split_images": dict(Counter(image_split.values())), "split_objects": dict(split_obj)},
    "eda": {
        "n_unique_images": int(len(unique_images)),
        "n_in_scope_objects": int(len(eda_df)),
        "image_width_mean": float(np.mean(widths)),
        "image_height_mean": float(np.mean(heights)),
        "objects_per_image_mean": float(np.mean(objs_per_image)),
        "bbox_area_px_p5": float(eda_df["area_px"].quantile(0.05)),
        "bbox_area_px_p50": float(eda_df["area_px"].quantile(0.50)),
        "modes": dict(modes_all),
        "invalid_boxes": invalid,
    },
}
meta_path = BASE_DIR / "dataset_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Wrote", meta_path)

# Copy metadata into reports for the Streamlit Performance page
reports = PROJECT_ROOT / "reports"
reports.mkdir(parents=True, exist_ok=True)
(reports / "dataset_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Also copied to", reports / "dataset_metadata.json")

In [ ]:
## Verify on-disk counts (do not trust memory only)

print("=== classification file counts ===")
for split in ("train", "val", "test"):
    total = 0
    for name in CLASS_NAMES:
        n = len(list((BASE_DIR / "classification" / split / name).glob("*.jpg")))
        total += n
    print(f"  {split:5s} {total}")

print("=== detection file counts ===")
for split in ("train", "val", "test"):
    ni = len(list((BASE_DIR / "detection" / "images" / split).glob("*.jpg")))
    nl = len(list((BASE_DIR / "detection" / "labels" / split).glob("*.txt")))
    print(f"  {split:5s} images={ni} labels={nl}")

# Spot-check one YOLO label
lab_dir = BASE_DIR / "detection" / "labels" / "train"
sample_lab = next(lab_dir.glob("*.txt"))
print("\nSample label", sample_lab.name, ":\n", sample_lab.read_text()[:400])
print("\nDataset ready at", BASE_DIR.resolve())

In [ ]:
## Optional: zip to Google Drive for the training notebooks

if IN_COLAB:
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    print("Zipping dataset to", zip_path, "(this can take a few minutes)...")
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", BASE_DIR)
    fig_zip = Path("/content/drive/MyDrive/smartvision_eda_figures.zip")
    shutil.make_archive(str(fig_zip.with_suffix("")), "zip", FIGURES_DIR)
    print("Done. Next: notebooks/02_classification_training.ipynb on a T4 GPU.")
else:
    print("Local run complete. Dataset at", BASE_DIR)